# Dermatology arm, step 1: intake (no models)

Downloads four open skin-lesion sources, builds manifests, maps diagnoses to a common scheme, and checks lesion/patient structure and duplicate images. **No model sees these images here**; the lock is written from these counts before any training.

| Source | Route | Modality |
|---|---|---|
| HAM10000 | ISIC archive public API (collection found by name) | dermoscopy |
| BCN20000 | ISIC archive public API | dermoscopy |
| Hospital Italiano de Buenos Aires | ISIC archive public API | dermoscopy |
| PAD-UFES-20 | Mendeley Data zip | smartphone clinical photos |

Outputs in `stochastic_orthogonality/data/derm/`: 512-pixel JPEGs (quality 95) per source, `manifest_<source>.csv`, `derm_intake_record.json`. CPU, High-RAM; 1 to 2 hours (mostly downloading). Resumable.

In [2]:
# Cell 1: setup
from google.colab import drive
drive.mount('/content/drive')
import os, io, json, zipfile, hashlib, datetime, time, glob, re
import requests, numpy as np, pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
OUT = '/content/drive/MyDrive/stochastic_orthogonality/data/derm'; os.makedirs(OUT, exist_ok=True)
RAW = '/content/derm_raw'; os.makedirs(RAW, exist_ok=True)
API = 'https://api.isic-archive.com/api/v2'
PAD_ZIP_URL = 'https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/zr7vgbcyr2-1.zip'
S = requests.Session(); S.headers.update({'User-Agent': 'research-intake'})
def get_json(url, params=None, tries=5):
    for i in range(tries):
        try:
            r = S.get(url, params=params, timeout=60); r.raise_for_status(); return r.json()
        except Exception as e:
            if i == tries - 1: raise
            time.sleep(2 * (i + 1))
print('ready')

Mounted at /content/drive
ready


In [3]:
# Cell 2: find the ISIC collections by name
cols, url = [], f'{API}/collections/'; params = {'limit': 100}
while url:
    j = get_json(url, params); cols += j['results']; url, params = j.get('next'), None
C = pd.DataFrame([{'id': c['id'], 'name': c['name'], 'public': c.get('public'), 'doi': (c.get('doi') or {}).get('url') if isinstance(c.get('doi'), dict) else c.get('doi')} for c in cols])
WANT = {'HAM10000': r'ham10000', 'BCN20000': r'bcn.?20000', 'HIBA': r'hospital italiano'}
FOUND = {}
for key, pat in WANT.items():
    hits = C[C.name.str.lower().str.contains(pat, regex=True)]
    print(f'\n{key}: candidate collections'); print(hits.to_string(index=False) if len(hits) else '  NONE FOUND')
    if len(hits): FOUND[key] = int(hits.iloc[0].id)
print('\nusing:', FOUND, '(edit FOUND here if a different candidate is the right one)')


HAM10000: candidate collections
 id     name  public  doi
212 HAM10000    True None

BCN20000: candidate collections
 id     name  public  doi
249 BCN20000    True None

HIBA: candidate collections
 id                                                                name  public             doi
251 Hospital Italiano de Buenos Aires - Skin Lesions Images (2019-2022)    True 10.34970/587329
176                      Hospital Italiano de Buenos Aires Skin Lesions    True 10.34970/432362

using: {'HAM10000': 212, 'BCN20000': 249, 'HIBA': 251} (edit FOUND here if a different candidate is the right one)


In [4]:
# Cell 3: download ISIC images and metadata (resumable, threaded)
def flatten(d, prefix=''):
    out = {}
    for k, v in (d or {}).items():
        if isinstance(v, dict): out.update(flatten(v, f'{prefix}{k}.'))
        else: out[f'{prefix}{k}'] = v
    return out
def fetch_collection(key, cid):
    meta_path = f'{RAW}/{key}_metadata.csv'
    if os.path.exists(meta_path): M = pd.read_csv(meta_path)
    else:
        rows, url, params = [], f'{API}/images/search/', {'collections': cid, 'limit': 100}
        while url:
            j = get_json(url, params); params = None
            for r in j['results']:
                rows.append({'isic_id': r['isic_id'], 'url': ((r.get('files') or {}).get('full') or {}).get('url'), 'license': r.get('copyright_license'),
                             'attribution': r.get('attribution'), **flatten(r.get('metadata'), 'm.')})
            url = j.get('next')
            if len(rows) % 2000 < 100: print(f'  {key}: {len(rows)} records')
        M = pd.DataFrame(rows); M.to_csv(meta_path, index=False)
    d = f'{RAW}/{key}'; os.makedirs(d, exist_ok=True)
    def dl(row):
        p = f'{d}/{row.isic_id}.jpg'
        if os.path.exists(p) and os.path.getsize(p) > 1000 or not isinstance(row.url, str): return
        for i in range(4):
            try:
                r = S.get(row.url, timeout=120); r.raise_for_status(); open(p, 'wb').write(r.content); return
            except Exception: time.sleep(2 * (i + 1))
    with ThreadPoolExecutor(16) as ex: list(ex.map(dl, M.itertuples()))
    got = sum(os.path.exists(f'{d}/{i}.jpg') for i in M.isic_id)
    print(f'{key}: {len(M)} records, {got} images downloaded'); return M
META = {k: fetch_collection(k, cid) for k, cid in FOUND.items()}

  HAM10000: 2000 records
  HAM10000: 4000 records
  HAM10000: 6000 records
  HAM10000: 8000 records
  HAM10000: 10000 records


HAM10000: 11720 records, 11720 images downloaded
  BCN20000: 2000 records
  BCN20000: 4000 records
  BCN20000: 6000 records
  BCN20000: 8000 records
  BCN20000: 10000 records
  BCN20000: 12000 records
  BCN20000: 14000 records
  BCN20000: 16000 records
  BCN20000: 18000 records


BCN20000: 18946 records, 18946 images downloaded


HIBA: 1616 records, 1616 images downloaded


In [8]:
# Cell 4 (revised): PAD-UFES-20 via Mendeley public API, with a manual-download fallback
pd_dir = f'{RAW}/PAD'; os.makedirs(pd_dir, exist_ok=True)
MANUAL_ZIP = f'{OUT}/manual/PAD-UFES-20.zip'      # fallback: put the "Download All" zip here
def have_pad(): return bool(glob.glob(f'{pd_dir}/**/metadata.csv', recursive=True)) and len(glob.glob(f'{pd_dir}/**/*.png', recursive=True)) > 2000
if not have_pad():
    try:
        files = get_json('https://data.mendeley.com/public-api/datasets/zr7vgbcyr2/files', {'folder_id': 'root', 'version': 1})
        for f in files:
            name, url = f['filename'], f['content_details']['download_url']
            dst = f'{pd_dir}/{name}'
            if not os.path.exists(dst):
                print('downloading', name)
                with S.get(url, timeout=1200, stream=True) as r:
                    r.raise_for_status()
                    with open(dst, 'wb') as fh:
                        for chunk in r.iter_content(1 << 22): fh.write(chunk)
    except Exception as e:
        print('Mendeley API route failed:', repr(e)[:200])
        assert os.path.exists(MANUAL_ZIP), f'Download the zip by hand (see instructions) and place it at {MANUAL_ZIP}'
        with zipfile.ZipFile(MANUAL_ZIP) as z: z.extractall(pd_dir)
    for inner in glob.glob(f'{pd_dir}/**/*.zip', recursive=True):      # image parts are nested zips
        with zipfile.ZipFile(inner) as z: z.extractall(os.path.dirname(inner))
PAD = pd.read_csv(glob.glob(f'{pd_dir}/**/metadata.csv', recursive=True)[0])
PAD_IMG = {os.path.basename(p): p for p in glob.glob(f'{pd_dir}/**/*.png', recursive=True)}
print('PAD-UFES-20:', len(PAD), 'rows;', len(PAD_IMG), 'png files; diagnostic counts', PAD.diagnostic.value_counts().to_dict())

Mendeley API route failed: HTTPError('403 Client Error: Forbidden for url: https://data.mendeley.com/public-api/datasets/zr7vgbcyr2/files?folder_id=root&version=1')
PAD-UFES-20: 2298 rows; 2298 png files; diagnostic counts {'BCC': 845, 'ACK': 730, 'NEV': 244, 'SEK': 235, 'SCC': 192, 'MEL': 52}


In [9]:
# Cell 5: common diagnosis scheme (unmapped values are printed, never guessed)
CLASSES = {'MEL': 'melanoma', 'BCC': 'basal cell carcinoma', 'SCC': 'squamous cell carcinoma incl. Bowen/in situ', 'AK': 'actinic keratosis',
           'NV': 'melanocytic nevus', 'BKL': 'benign keratosis (seborrheic keratosis, solar lentigo, lichen planus-like)', 'DF': 'dermatofibroma', 'VASC': 'vascular', 'OTHER': 'other'}
RULES = [('MEL', r'melanoma'), ('BCC', r'basal cell'), ('SCC', r'squamous cell|bowen|in situ, squamous|keratoacanthoma'), ('AK', r'actinic keratosis|akiec'),
         ('NV', r'nev(us|i)'), ('BKL', r'seborrh|lentigo|lichen|keratosis-like|benign keratosis'), ('DF', r'dermatofibroma'),
         ('VASC', r'angioma|angiokeratoma|pyogenic|hemorrhage|vascular')]
def map_dx(text):
    t = str(text).lower()
    for cls, pat in RULES:
        if re.search(pat, t): return cls
    return None
def isic_manifest(key, M):
    dxcols = [c for c in M.columns if re.search(r'diagnosis', c, re.I)]
    text = M[dxcols].astype(str).agg(' | '.join, axis=1) if dxcols else pd.Series([''] * len(M))
    cls = text.map(map_dx)
    lcol = next((c for c in M.columns if c.endswith('lesion_id')), None); pcol = next((c for c in M.columns if c.endswith('patient_id')), None)
    bm = next((c for c in M.columns if c.endswith('benign_malignant')), None)
    df = pd.DataFrame({'image_id': M.isic_id, 'source': key, 'modality': 'dermoscopy', 'dx_text': text, 'dx': cls,
                       'lesion_id': M[lcol] if lcol else None, 'patient_id': M[pcol] if pcol else None, 'benign_malignant': M[bm] if bm else None,
                       'license': M.license, 'path': [f'{RAW}/{key}/{i}.jpg' for i in M.isic_id]})
    return df
MAN = {k: isic_manifest(k, M) for k, M in META.items()}
PADMAP = {'MEL': 'MEL', 'BCC': 'BCC', 'SCC': 'SCC', 'BOD': 'SCC', 'ACK': 'AK', 'NEV': 'NV', 'SEK': 'BKL'}
MAN['PAD'] = pd.DataFrame({'image_id': PAD.img_id.str.replace('.png', '', regex=False), 'source': 'PAD', 'modality': 'smartphone', 'dx_text': PAD.diagnostic,
                           'dx': PAD.diagnostic.map(PADMAP), 'lesion_id': PAD.lesion_id.astype(str), 'patient_id': PAD.patient_id.astype(str),
                           'benign_malignant': None, 'license': 'CC BY 4.0 (Mendeley)', 'path': [PAD_IMG.get(i) for i in PAD.img_id]})
for k, df in MAN.items():
    df['malignant'] = df.dx.isin(['MEL', 'BCC', 'SCC']); df['melanoma'] = df.dx.eq('MEL')
    un = df[df.dx.isna()].dx_text.value_counts()
    print(f'\n{k}: {len(df)} images | lesion_id present: {df.lesion_id.notna().mean():.0%} | patient_id present: {df.patient_id.notna().mean():.0%}')
    print('  classes:', df.dx.value_counts().to_dict())
    if len(un): print('  UNMAPPED diagnosis strings (review before the lock):'); print(un.head(25).to_string())


HAM10000: 11720 images | lesion_id present: 100% | patient_id present: 0%
  classes: {'NV': 7737, 'BKL': 1338, 'MEL': 1305, 'BCC': 622, 'SCC': 229, 'VASC': 180, 'DF': 160, 'AK': 149}

BCN20000: 18946 images | lesion_id present: 100% | patient_id present: 0%
  classes: {'NV': 5647, 'MEL': 4636, 'BCC': 3676, 'BKL': 1551, 'AK': 1088, 'SCC': 559, 'DF': 168, 'VASC': 151}
  UNMAPPED diagnosis strings (review before the lock):
dx_text
nan | nan | nan | histopathology                                                                                         870
nan | nan | nan | nan                                                                                                    218
Benign | Benign soft tissue proliferations - Fibro-histiocytic | Scar | histopathology                                   160
Benign | Benign soft tissue proliferations - Fibro-histiocytic | Scar | nan                                              107
nan | nan | nan | confocal microscopy with consensus dermoscopy    

In [ ]:
# Cell 6: duplicates (byte hashes, within and across sources), 512-pixel copies, manifests, record
def sha(p):
    h = hashlib.sha256(); h.update(open(p, 'rb').read()); return h.hexdigest()
ALL = pd.concat(MAN.values(), ignore_index=True)
ALL = ALL[ALL.path.notna() & ALL.path.map(lambda p: isinstance(p, str) and os.path.exists(p))].copy()
with ThreadPoolExecutor(16) as ex: ALL['raw_sha256'] = list(ex.map(sha, ALL.path))
dup = ALL[ALL.duplicated('raw_sha256', keep=False)]
cross = dup.groupby('raw_sha256').source.nunique(); cross_n = int((cross > 1).sum())
print(f'byte-identical groups: {dup.raw_sha256.nunique()} ({len(dup)} images); groups spanning more than one source: {cross_n}')
def to512(row):
    d = f'{OUT}/images_512/{row.source}'; os.makedirs(d, exist_ok=True); p = f'{d}/{row.image_id}.jpg'
    if not os.path.exists(p):
        im = Image.open(row.path).convert('RGB'); w, h = im.size; s = 512 / min(w, h)
        im = im.resize((max(512, round(w * s)), max(512, round(h * s))), Image.LANCZOS); w, h = im.size
        im.crop(((w - 512) // 2, (h - 512) // 2, (w - 512) // 2 + 512, (h - 512) // 2 + 512)).save(p, 'JPEG', quality=95)
    return p
with ThreadPoolExecutor(8) as ex: ALL['path_512'] = list(ex.map(to512, ALL.itertuples()))
for k in MAN:
    ALL[ALL.source == k].drop(columns=['path']).to_csv(f'{OUT}/manifest_{k}.csv', index=False)
rec = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'isic_collections': FOUND,
       'preprocessing': 'RGB, shorter side resized to 512 (Lanczos), centre crop 512x512, JPEG quality 95',
       'sources': {k: {'images': int((ALL.source == k).sum()), 'lesions': int(ALL[ALL.source == k].lesion_id.nunique()),
                       'patients': int(ALL[ALL.source == k].patient_id.nunique()), 'classes': ALL[ALL.source == k].dx.value_counts().to_dict(),
                       'malignant': int(ALL[ALL.source == k].malignant.sum()), 'melanoma': int(ALL[ALL.source == k].melanoma.sum()),
                       'unmapped': int(ALL[ALL.source == k].dx.isna().sum())} for k in MAN},
       'byte_duplicate_groups': int(dup.raw_sha256.nunique()), 'cross_source_duplicate_groups': cross_n,
       'note': 'intake only; no model has seen these images; lock to be written from these counts'}
json.dump(rec, open(f'{OUT}/derm_intake_record.json', 'w'), indent=2, default=str); print(json.dumps(rec, indent=1, default=str))

byte-identical groups: 15 (30 images); groups spanning more than one source: 0
{
 "generated_utc": "2026-09-26T20:15:40.533687+00:00",
 "isic_collections": {
  "HAM10000": 212,
  "BCN20000": 249,
  "HIBA": 251
 },
 "preprocessing": "RGB, shorter side resized to 512 (Lanczos), centre crop 512x512, JPEG quality 95",
 "sources": {
  "HAM10000": {
   "images": 11720,
   "lesions": 8838,
   "patients": 0,
   "classes": {
    "NV": 7737,
    "BKL": 1338,
    "MEL": 1305,
    "BCC": 622,
    "SCC": 229,
    "VASC": 180,
    "DF": 160,
    "AK": 149
   },
   "malignant": 2156,
   "melanoma": 1305,
   "unmapped": 0
  },
  "BCN20000": {
   "images": 18946,
   "lesions": 5440,
   "patients": 0,
   "classes": {
    "NV": 5647,
    "MEL": 4636,
    "BCC": 3676,
    "BKL": 1551,
    "AK": 1088,
    "SCC": 559,
    "DF": 168,
    "VASC": 151
   },
   "malignant": 8871,
   "melanoma": 4636,
   "unmapped": 1470
  },
  "HIBA": {
   "images": 1616,
   "lesions": 1246,
   "patients": 623,
   "classes": {
